#

In [1]:
# imports
import pandas as pd
import os

# Helpers
from Scripts.q1k_preprocessing import create_diagnosis_columns, create_IQ_column

In [2]:
root_dir = '/Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL'
database = pd.read_csv(os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv.csv'))

# Table of Contents
1. [Create new diagnosis columns](#Create-new-diagnosis-columns)
2. [Sample Overview](#sample-overview)

# Create new diagnosis columns

In [3]:
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols.csv')

# Apply the function to create new diagnosis columns
create_diagnosis_columns(database, output_path)

database_clean_cols = pd.read_csv(output_path)

# Display the counts for each new diagnosis column
diagnosis_columns = [
    'ASD', 'ASD_behavior', 'ADHD', 'ID', 'OCD', 'motor_disorder',
    'anxiety', 'neurological_conditions', 'genetic_disorder', 'other'
]

print("\nDiagnosis Distribution:")
for col in diagnosis_columns:
    count = database_clean_cols[col].sum()
    percentage = (count / len(database) * 100).round(1)
    print(f"{col}: {count} ({percentage}%)")



Database with new columns saved to: /Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols.csv

Diagnosis Distribution:
ASD: 51.0 (23.8%)
ASD_behavior: 20.0 (9.3%)
ADHD: 70.0 (32.7%)
ID: 25.0 (11.7%)
OCD: 0.0 (0.0%)
motor_disorder: 6.0 (2.8%)
anxiety: 26.0 (12.1%)
neurological_conditions: 37 (17.3%)
genetic_disorder: 8.0 (3.7%)
other: 85 (39.7%)


In [4]:
output_path = os.path.join(root_dir, 'Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ.csv')

# Apply the function to create new diagnosis columns
create_IQ_column(database_clean_cols, output_path)

database_clean_cols_IQ = pd.read_csv(output_path)

Database with IQ column saved to: /Users/emmanuelle.coutu-nadeau/Library/Mobile Documents/com~apple~CloudDocs/UdeM/MSc Psycho/LABO NED - Personal Drive/Code/GENiAL/Data/Q1KDatabase-ECNEEGIQGENCHUSJ_DATA_flattened_cleaned_cnv_renamedcols_IQ.csv


# Sample Overview

In [7]:
# Create a function to calculate counts and percentages for each group
def get_group_stats(df, group_col, diagnosis_col):
    total = df[group_col].sum()
    count = df[(df[group_col] == 1) & (df[diagnosis_col] == 1)].shape[0]
    percentage = (count / total * 100).round(1) if total > 0 else 0
    return count, percentage, total

# Create a table to store results
results = []

# For each diagnosis column
for diagnosis in diagnosis_columns:
    row = {'Diagnosis': diagnosis}
    
    # Get stats for Control group
    count, pct, total = get_group_stats(database_clean_cols_IQ, 'Control', diagnosis)
    row['Control'] = f"{count} ({pct}%)"
    
    # Get stats for Neurodev without Genetic_carrier
    neurodev_no_genetic = database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 0)]
    total = len(neurodev_no_genetic)
    count = neurodev_no_genetic[diagnosis].sum()
    pct = (count / total * 100).round(1) if total > 0 else 0
    row['Neurodev without Genetic_carrier'] = f"{count} ({pct}%)"
    
    # Get stats for Neurodev AND Genetic_carrier
    neurodev_genetic = database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 1)]
    total = len(neurodev_genetic)
    count = neurodev_genetic[diagnosis].sum()
    pct = (count / total * 100).round(1) if total > 0 else 0
    row['Neurodev AND Genetic_carrier'] = f"{count} ({pct}%)"
    
    results.append(row)

# Convert to DataFrame and display
results_df = pd.DataFrame(results)

# Add sample sizes to column headers
control_total = database_clean_cols_IQ['Control'].sum()
neurodev_no_genetic_total = len(database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 0)])
neurodev_genetic_total = len(database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 1)])

results_df.columns = ['Diagnosis', 
                     f'Control [n={control_total}]', 
                     f'Neurodev without Genetic_carrier [n={neurodev_no_genetic_total}]', 
                     f'Neurodev AND Genetic_carrier [n={neurodev_genetic_total}]']

print("\nDiagnosis Distribution by Group:")
display(results_df)

# Add age, sex, and IQ information for each group
print("\nAge, Sex, and IQ Distribution by Group:")
demographic_results = []

for group in ['Control', 'Neurodev without Genetic_carrier', 'Neurodev AND Genetic_carrier']:
    if group == 'Neurodev AND Genetic_carrier':
        group_data = database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 1)]
    elif group == 'Neurodev without Genetic_carrier':
        group_data = database_clean_cols_IQ[(database_clean_cols_IQ['Neurodev'] == 1) & (database_clean_cols_IQ['Genetic_carrier'] == 0)]
    else:
        group_data = database_clean_cols_IQ[database_clean_cols_IQ[group] == 1]
    
    # Calculate age statistics
    age_mean = group_data['eeg_test_age'].mean()
    age_std = group_data['eeg_test_age'].std()
    
    # Calculate sex distribution
    sex_counts = group_data['sex'].value_counts()
    total = len(group_data)
    male_pct = (sex_counts.get('M', 0) / total * 100).round(1)
    female_pct = (sex_counts.get('F', 0) / total * 100).round(1)
    
    # Calculate IQ statistics
    iq_mean = group_data['IQ'].mean()
    iq_std = group_data['IQ'].std()
    
    demographic_results.append({
        'Group': group,
        'Age (mean ± std)': f"{age_mean:.1f} ± {age_std:.1f}",
        'Male': f"{sex_counts.get('M', 0)} ({male_pct}%)",
        'Female': f"{sex_counts.get('F', 0)} ({female_pct}%)",
        'IQ (mean ± std)': f"{iq_mean:.1f} ± {iq_std:.1f}"
    })

# Display demographic results
demographic_df = pd.DataFrame(demographic_results)
display(demographic_df)



Diagnosis Distribution by Group:


,Diagnosis,Control [n=63.0],Neurodev without Genetic_carrier [n=71],Neurodev AND Genetic_carrier [n=54]
0,ASD,0 (0.0%),23.0 (32.4%),24.0 (44.4%)
1,ASD_behavior,1 (1.6%),8.0 (11.3%),10.0 (18.5%)
2,ADHD,0 (0.0%),36.0 (50.7%),27.0 (50.0%)
3,ID,0 (0.0%),4.0 (5.6%),20.0 (37.0%)
4,OCD,0 (0.0%),0.0 (0.0%),0.0 (0.0%)
5,motor_disorder,0 (0.0%),4.0 (5.6%),2.0 (3.7%)
6,anxiety,1 (1.6%),17.0 (23.9%),7.0 (13.0%)
7,neurological_conditions,0 (0.0%),12 (16.9%),23 (42.6%)
8,genetic_disorder,0 (0.0%),0.0 (0.0%),7.0 (13.0%)
9,other,1 (1.6%),32 (45.1%),45 (83.3%)



Age, Sex, and IQ Distribution by Group:


,Group,Age (mean ± std),Male,Female,IQ (mean ± std)
0,Control,32.9 ± 16.6,27 (42.9%),36 (57.1%),107.0 ± 13.8
1,Neurodev without Genetic_carrier,23.3 ± 15.4,36 (50.7%),35 (49.3%),104.1 ± 19.4
2,Neurodev AND Genetic_carrier,11.7 ± 8.0,31 (57.4%),23 (42.6%),80.2 ± 22.2
